# SAM Auto-Label das 22 fotos magenta

Usa Segment Anything (Meta) guiado pela `_magenta_plant_mask` do projeto para gerar labels YOLO precisas em fotos magenta extremo, que o modelo atual rotula errado.

**Pre-requisito:** sessao Colab com T4 ativa e `/content/repo` clonado. Se ja rodou as celulas 1-3 do colab_train.ipynb, esta pronto.

Saida: labels YOLO em `/content/sam_labels/` + dashboard HTML pra revisao.

## 1. Garantir repo + dependencias

In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone --depth 1 https://github.com/nikolasdehor/projetogerminacao.git /content/repo
%cd /content/repo
!pip install -q 'segment-anything-py>=1.0' 'torch>=2.3' 'opencv-python-headless>=4.10' 'numpy>=1.26' 'Pillow>=10.4'

## 2. Baixar checkpoint do SAM (vit_b, ~375MB, ~30s no Colab)

In [ ]:
import urllib.request, os
CKPT = '/content/sam_vit_b.pth'
if not os.path.exists(CKPT):
    print('Baixando SAM vit_b...')
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        CKPT,
    )
print(f'OK: {os.path.getsize(CKPT) / 1e6:.0f}MB')

## 3. Carregar SAM em GPU

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
sam = sam_model_registry['vit_b'](checkpoint='/content/sam_vit_b.pth')
sam.to(device)
predictor = SamPredictor(sam)
print('SAM pronto')

## 4. Funcoes auxiliares (mascara magenta do projeto + filtros)

In [ ]:
import cv2
import numpy as np
from pathlib import Path

def magenta_plant_mask(img_bgr):
    """Igual a app.inference._magenta_plant_mask. Pixels de planta em LED magenta."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h_ch = hsv[:, :, 0]
    v_ch = hsv[:, :, 2]
    g_ch = img_bgr[:, :, 1]
    mask = (
        (h_ch >= 125) & (h_ch <= 175)
        & (g_ch >= 28)
        & (v_ch >= 110)
        & (v_ch <= 245)
    ).astype(np.uint8) * 255
    # Limpa ruido
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def candidate_bboxes(mask, img_shape, min_area=400, max_area_ratio=0.05, min_dim=15):
    """Connected components da mascara -> bboxes candidatas a planta."""
    h, w = img_shape[:2]
    max_area = h * w * max_area_ratio
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    boxes = []
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if area < min_area or area > max_area:
            continue
        if bw < min_dim or bh < min_dim:
            continue
        # Aspect ratio plausivel pra planta
        ar = bw / max(bh, 1)
        if ar < 0.3 or ar > 3.5:
            continue
        boxes.append((x, y, x + bw, y + bh))
    return boxes

## 5. Processar as 22 fotos magenta

Para cada foto:
1. Calcula mascara magenta -> bboxes candidatas
2. Pra cada bbox, pede ao SAM uma mascara refinada
3. Extrai bbox final da mascara SAM
4. Salva como YOLO label (class 0 = Germinacao)

In [ ]:
PENDING = Path('/content/repo/dataset/pending_manual_label/images')
OUT_DIR = Path('/content/sam_labels')
OUT_DIR.mkdir(exist_ok=True)
OUT_LABELS = OUT_DIR / 'labels'
OUT_PREVIEW = OUT_DIR / 'previews'
OUT_LABELS.mkdir(exist_ok=True)
OUT_PREVIEW.mkdir(exist_ok=True)

images = sorted(PENDING.glob('*.jpeg')) + sorted(PENDING.glob('*.jpg')) + sorted(PENDING.glob('*.png'))
print(f'Fotos pra processar: {len(images)}')

all_stats = []
for idx, img_path in enumerate(images, 1):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Step 1: mascara magenta -> candidatas
    mask = magenta_plant_mask(img_bgr)
    candidates = candidate_bboxes(mask, img_bgr.shape)

    # Step 2: SAM refina cada candidato
    predictor.set_image(img_rgb)
    final_boxes = []
    for box in candidates:
        masks, scores, _ = predictor.predict(
            box=np.array(box),
            multimask_output=False,
        )
        sam_mask = masks[0].astype(np.uint8) * 255
        # Sanity: mascara SAM tem que ter overlap com a mascara magenta original
        overlap = cv2.bitwise_and(sam_mask, mask)
        overlap_pct = float(cv2.countNonZero(overlap)) / max(cv2.countNonZero(sam_mask), 1)
        if overlap_pct < 0.3:
            continue
        ys, xs = np.where(sam_mask > 0)
        if ys.size == 0:
            continue
        x1f, y1f = int(xs.min()), int(ys.min())
        x2f, y2f = int(xs.max()) + 1, int(ys.max()) + 1
        # Pad leve
        pad_x, pad_y = int((x2f - x1f) * 0.05), int((y2f - y1f) * 0.05)
        x1f, y1f = max(0, x1f - pad_x), max(0, y1f - pad_y)
        x2f, y2f = min(w, x2f + pad_x), min(h, y2f + pad_y)
        final_boxes.append((x1f, y1f, x2f, y2f, float(scores[0])))

    # Step 3: salva YOLO label (class 0 = Germinacao)
    lbl_path = OUT_LABELS / (img_path.stem + '.txt')
    with open(lbl_path, 'w') as f:
        for x1f, y1f, x2f, y2f, _ in final_boxes:
            cx = (x1f + x2f) / 2.0 / w
            cy = (y1f + y2f) / 2.0 / h
            bw_n = (x2f - x1f) / w
            bh_n = (y2f - y1f) / h
            f.write(f'0 {cx:.6f} {cy:.6f} {bw_n:.6f} {bh_n:.6f}\n')

    # Step 4: preview com bboxes
    preview = img_bgr.copy()
    for i, (x1f, y1f, x2f, y2f, score) in enumerate(final_boxes, 1):
        cv2.rectangle(preview, (x1f, y1f), (x2f, y2f), (0, 255, 0), 2)
        cv2.putText(preview, f'{i}', (x1f, max(15, y1f - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
    cv2.imwrite(str(OUT_PREVIEW / img_path.name), preview)

    all_stats.append((img_path.name, len(candidates), len(final_boxes)))
    print(f'[{idx}/{len(images)}] {img_path.name}: {len(candidates)} candidatas -> {len(final_boxes)} aceitas')

print()
print('=== RESUMO ===')
total_boxes = sum(s[2] for s in all_stats)
print(f'Total fotos processadas: {len(all_stats)}')
print(f'Total bboxes geradas:    {total_boxes}')
print(f'Media por foto:          {total_boxes / max(len(all_stats), 1):.1f}')

## 6. Gerar HTML de revisao com as previews

Voce abre o HTML e olha rapido foto por foto. Se uma estiver claramente errada, anota o nome pra rejeitar depois.

In [ ]:
import base64
from pathlib import Path

previews = sorted(Path('/content/sam_labels/previews').glob('*'))
html = ['<!doctype html><html><head><meta charset="utf-8"><title>SAM Review</title>',
        '<style>body{background:#111;color:#eee;font-family:sans-serif;padding:20px}',
        '.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(380px,1fr));gap:20px}',
        '.card{background:#1a1a1a;padding:8px;border-radius:6px;border:1px solid #333}',
        '.card img{width:100%;display:block;border-radius:4px}',
        '.fn{font-family:monospace;font-size:11px;word-break:break-all;margin-top:6px;color:#aaa}',
        '</style></head><body>',
        f'<h1>SAM Auto-Labels - {len(previews)} fotos magenta</h1><div class="grid">']
for p in previews:
    img_bytes = p.read_bytes()
    img_b64 = base64.b64encode(img_bytes).decode()
    n_boxes = len((Path('/content/sam_labels/labels') / (p.stem + '.txt')).read_text().strip().split('\n')) if (Path('/content/sam_labels/labels') / (p.stem + '.txt')).exists() else 0
    html.append(f'<div class="card"><img src="data:image/jpeg;base64,{img_b64}"><div class="fn">{p.name} ({n_boxes} bbox)</div></div>')
html.append('</div></body></html>')
out_html = Path('/content/sam_review.html')
out_html.write_text(''.join(html))
print(f'HTML salvo em: {out_html} ({out_html.stat().st_size / 1024:.0f}KB)')
print('Baixe com a celula 7 abaixo.')

## 7. Baixar HTML + labels pro Mac

In [ ]:
!cd /content && tar -czf sam_labels.tar.gz sam_labels sam_review.html
from google.colab import files
files.download('/content/sam_labels.tar.gz')